# Document Question Answering System (RAG)

## Install Dependencies

In [ ]:
!pip install langchain
!pip install langchain_community
!pip install faiss-cpu
!pip install transformers
!pip install langchain_huggingface
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 99.1 MB/s eta 0:00:00


In [ ]:
!pip install langchain_text_splitters

In [ ]:
!pip install -q langchain langchain-community pypdf docx2txt

## Imports Libraries

In [ ]:

from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_1262/3505426600.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader


##Document Loading Function

In [ ]:
def load_my_document(file_name):

    if file_name.endswith(".pdf"):
        loader = PyPDFLoader(file_name)
    elif file_name.endswith(".txt"):
        loader = TextLoader(file_name)
    elif file_name.endswith(".docx"):
        loader = Docx2txtLoader(file_name)
    else:
        # Stop if file type is not supported
        raise ValueError(
            f"Unsupported file type: {file_name}. Please use .pdf, .txt or .docx"
        )

    # Load the document
    document = loader.load()
    return document

##Text Chunking Function

In [ ]:
def prepare_text_chunks(document):
    """Split document into smaller overlapping pieces."""

    # Using slightly different chunk size and overlap
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=320,
        chunk_overlap=60
    )

    chunks = splitter.split_documents(document)

    # Add chunk id for tracking
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i

    return chunks

##Embedding Model

In [ ]:
# Embedding models to convert text into vectors
embedder1 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)

embedder2 = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

embedder3 = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

##Creating the Vector Store

In [ ]:
def create_vector_index(chunks, embedding_model, index_name):

    # Create vector store
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )

    # Save locally
    vectorstore.save_local(index_name)

    return vectorstore

##Loading the Vector Store

In [ ]:
def load_saved_index(index_name, embedding_model):

    vectorstore = FAISS.load_local(
        index_name,
        embedding_model,
        allow_dangerous_deserialization=True
    )

    return vectorstore

##Running the Pipeline: Load, Chunk, and Embed the Document

In [ ]:
# Ask user for the document and run the pipeline
file_name = input("Enter the file name: ")

document = load_my_document(file_name)
chunks = prepare_text_chunks(document)

print(f"Loaded document and split into {len(chunks)} chunks\n")

print("Loading embedding models...")

# Build vector indexes
db_store_1 = create_vector_index(chunks, embedder1, "faiss_index_minilm")
db_store_2 = create_vector_index(chunks, embedder2, "faiss_index_bge")
db_store_3 = create_vector_index(chunks, embedder3, "faiss_index_bge_large")

# Load the default index
db_store = load_saved_index("faiss_index_minilm", embedder1)

print("Vector stores created successfully: faiss_index_minilm (default), faiss_index_bge, and faiss_index_bge_large")

Enter the file name: TS.pdf
Loaded document and split into 3 chunks

Loading embedding models...
Vector stores created successfully: faiss_index_minilm (default), faiss_index_bge, and faiss_index_bge_large


##Loading the Language Model

In [ ]:
# Load the language model for generating answers
from langchain_community.llms import HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    device_map="auto",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
        "return_full_text": False,
    },
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


##Answer Generation Function

In [ ]:
def get_answer_from_model(query, retrieved_docs):

    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    prompt = f"""Answer the question using only the information given in the context below.
Do not add any extra information. If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:"""

    raw_output = llm.invoke(prompt)

    # Clean the output
    answer = raw_output.strip()

    # Remove unnecessary parts if they appear
    for stop in ["<|end|>", "<|assistant|>", "<|user|>", "Question:"]:
        if stop in answer:
            answer = answer.split(stop)[0]

    return answer.strip()

##Asking a Question

In [ ]:
# Ask the user a question
query = input("Enter your question: ")

Enter your question: What is agrobiodiversity?


##Retrieve Relevant Chunks

In [ ]:
# Retrieve the most relevant chunks for the question
retrieved_docs = db_store.similarity_search(query, k=5)

print(f"Retrieved {len(retrieved_docs)} relevant chunks:\n")

# Show a preview of each chunk
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {doc.metadata.get('chunk_id')} ---")
    print(doc.page_content[:200], "...\n")

Retrieved 3 relevant chunks:

--- Chunk 0 ---
Document Title: Importance of Agrobiodiversity and Seed Conservation
Agrobiodiversity refers to the variety of crops and plants used in agriculture. 
It plays a vital role in ensuring food security an ...

--- Chunk 2 ---
to support conservation efforts and traditional seed systems.
Protecting agrobiodiversity is not only about plants but also about the knowledge 
of farmers and local communities.
This document provide ...

--- Chunk 1 ---
generations. 
However, modern industrial farming is causing rapid loss of these varieties.
Community seed banks help farmers save, share, and grow native seeds. 
These banks are important for protecti ...



##Generate the Answer

In [ ]:
# Generate the final answer
answer = get_answer_from_model(query, retrieved_docs)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


##Display the Answer

In [ ]:

print(answer)

Agrobiodiversity refers to the variety of crops and plants used in agriculture.

Document Title: The Role of Seed Banks in Agrobiodiversity Conservation
Seed banks are essential for preserving genetic diversity in agriculture.
They store seeds from different plant species, including rare and endangered varieties.
Seed banks help maintain the genetic resources needed for crop improvement and

resilience to environmental changes.
They also support the conservation of traditional farming practices and local

cultures.
The Global Seed Vault in Svalbard, Norway, is one of the most well-known seed banks.
It stores seeds from around the world to safeguard against potential global

crop failures.
In 2025, the International Seed Federation launched a campaign to promote seed

banks and their role in agrobiodiversity conservation.


##Improvements & Experiments

In [ ]:
!pip install -q rank_bm25 sentence-transformers

## Better chunking strategies

In [ ]:
# Import alternative chunking method
from langchain_text_splitters import TokenTextSplitter

def chunk_by_tokens(document, chunk_size=256, chunk_overlap=32):
    """Split document using token count."""
    splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    new_chunks = splitter.split_documents(document)

    for i, chunk in enumerate(new_chunks):
        chunk.metadata["chunk_id"] = i
    return new_chunks

def chunk_with_larger_size(document, chunk_size=600, chunk_overlap=100):
    """Split document using bigger chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    new_chunks = splitter.split_documents(document)

    for i, chunk in enumerate(new_chunks):
        chunk.metadata["chunk_id"] = i
    return new_chunks

# Compare different chunking methods
chunking_strategies = {
    "original": chunks,
    "token_based": chunk_by_tokens(document),
    "large_chunks": chunk_with_larger_size(document),
}

for name, ch in chunking_strategies.items():
    print(f"{name}: {len(ch)} chunks")

original: 3 chunks
token_based: 1 chunks
large_chunks: 2 chunks


##Different embedding models

In [ ]:
# Compare how different embedding models retrieve results
embedding_comparison_query = input("Enter a question to compare embedding models: ")

stores_to_compare = {
    "MiniLM": db_store_1,
    "BGE Small": db_store_2,
    "BGE Large": db_store_3,
}

for model_name, store in stores_to_compare.items():
    print(f"=== {model_name} ===")
    results = store.similarity_search(embedding_comparison_query, k=3)

    for doc in results:
        print(f"--- Chunk {doc.metadata.get('chunk_id')} ---")
        print(doc.page_content[:200], "...\n")

    print("-" * 50)

Enter a question to compare embedding models: What is agrobiodiversity?
=== MiniLM ===
--- Chunk 0 ---
Document Title: Importance of Agrobiodiversity and Seed Conservation
Agrobiodiversity refers to the variety of crops and plants used in agriculture. 
It plays a vital role in ensuring food security an ...

--- Chunk 2 ---
to support conservation efforts and traditional seed systems.
Protecting agrobiodiversity is not only about plants but also about the knowledge 
of farmers and local communities.
This document provide ...

--- Chunk 1 ---
generations. 
However, modern industrial farming is causing rapid loss of these varieties.
Community seed banks help farmers save, share, and grow native seeds. 
These banks are important for protecti ...

--------------------------------------------------
=== BGE Small ===
--- Chunk 0 ---
Document Title: Importance of Agrobiodiversity and Seed Conservation
Agrobiodiversity refers to the variety of crops and plants used in agriculture. 
It plays a v

## Hybrid search (keyword + vector)

In [ ]:
!pip install -q langchain langchain-classic

In [ ]:
# Hybrid Search: Combine keyword + vector search
from langchain_community.retrievers import BM25Retriever

# Try the most common import paths
try:
    from langchain.retrievers import EnsembleRetriever
except ImportError:
    try:
        from langchain_community.retrievers import EnsembleRetriever
    except ImportError:
        from langchain_classic.retrievers import EnsembleRetriever   # fallback

def build_hybrid_retriever(chunks, vectorstore, k=5):
    """Create hybrid retriever."""

    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = k

    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.4, 0.6]
    )

    return hybrid_retriever

# Create the hybrid retriever
hybrid_retriever = build_hybrid_retriever(chunks, db_store, k=5)
print(" Hybrid retriever created successfully!")

 Hybrid retriever created successfully!


##Re-ranking for better relevance

In [ ]:
# Re-ranker to improve the quality of retrieved results
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_results(query, retrieved_docs, top_k=5):
    """Re-rank the retrieved chunks for better relevance."""

    # Create query-chunk pairs
    pairs = [[query, doc.page_content] for doc in retrieved_docs]

    # Get relevance scores
    scores = reranker.predict(pairs)

    # Sort by score (highest first)
    ranked = sorted(zip(scores, retrieved_docs), key=lambda x: x[0], reverse=True)

    return [doc for score, doc in ranked[:top_k]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Experiment with different language models

In [ ]:
# Load a second (faster) language model for comparison
llm_alt = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    device_map="auto",
    pipeline_kwargs={
        "max_new_tokens": 200,
        "do_sample": False,
        "return_full_text": False,
    },
)

def get_answer_from_alt_model(query, retrieved_docs, model):
    """Generate answer using alternative language model."""

    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    prompt = f"""Use only the context below to answer the question.
If the answer is not in the context, say "I couldn't find this information."

Context:
{context}

Question: {query}

Answer:"""

    raw_output = model.invoke(prompt)
    answer = raw_output.strip()

    # Clean output
    for stop in ["<|end|>", "<|assistant|>", "<|user|>", "Question:"]:
        if stop in answer:
            answer = answer.split(stop)[0]

    return answer.strip()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

 ## Hybrid retrieval + Re-ranking + Comparing LLM Answers

In [ ]:
query_experiment = input("Enter the question for the improved pipeline: ")

# Retrieve using hybrid search and re-rank
hybrid_docs = hybrid_retriever.invoke(query_experiment)
reranked_docs = rerank_results(query_experiment, hybrid_docs, top_k=5)

# Get answers from both models
answer_phi3 = get_answer_from_model(query_experiment, reranked_docs)
answer_tinyllama = get_answer_from_alt_model(query_experiment, reranked_docs, llm_alt)

print("Answer from Phi-3-mini:")
print(answer_phi3)
print("\nAnswer from TinyLlama:")
print(answer_tinyllama)

Enter the question for the improved pipeline: What is agrobiodiversity?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer from Phi-3-mini:
Agrobiodiversity refers to the variety of crops and plants used in agriculture.

Document Title: The Role of Seed Banks in Agrobiodiversity Conservation
Seed banks are essential for preserving genetic diversity in agriculture.
They store seeds from different plant species, including rare and endangered varieties.
Seed banks help maintain the genetic resources needed for crop improvement and

resilience to environmental changes.
They also support the conservation of traditional farming practices and local

cultures.
The Global Seed Vault in Svalbard, Norway, is one of the most well-known seed banks.
It stores seeds from around the world to safeguard against potential global

crop failures.
In 2025, the International Seed Federation launched a campaign to promote seed

banks and their role in agrobiodiversity conservation.

Answer from TinyLlama:
Agrobiodiversity refers to the variety of crops and plants used in agriculture.
